# 05 · Inference & Evaluation

Notebook 04 trained 3 fusion strategies x 2 tasks x every CV fold, saving the best-val-loss
checkpoint for each (fusion, task, fold) combination. This notebook loads those checkpoints back,
runs inference over each fold's held-out validation patients, aggregates the slide-level
predictions back to patient level, applies decision thresholds / risk stratification, and
produces the comparison metrics and visualizations for the whole course.

Every model here still only ever sees one slide per forward pass -- all of the "many slides, one
patient" logic lives in this notebook's dataset and aggregation code, exactly as it did in
notebook 04.

## Setup

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score
from scipy.ndimage import gaussian_filter
from lifelines import KaplanMeierFitter
from lifelines.statistics import logrank_test
from PIL import Image
import openslide

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.fusion_models import (
    EarlyFusionModel, IntermediateFusionModel, LateFusionModel, concordance_index,
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEVICE

In [ ]:
# --- Project paths -----------------------------------------------------------
LABELS_DIR = PROJECT_ROOT / "data" / "labels"
PREPARED_DIR = PROJECT_ROOT / "data" / "prepared"
FEATURES_DIR = PROJECT_ROOT / "data" / "features"
RAW_WSI_DIR = PROJECT_ROOT / "data" / "raw" / "task1" / "pathology" / "images"

CLINICAL_EMBEDDING_DIR = PREPARED_DIR / "clinical_embeddings"
WSI_FEATURE_DIR = FEATURES_DIR / "wsi"
MRI_FEATURE_DIR = FEATURES_DIR / "mri"
CHECKPOINT_DIR = PROJECT_ROOT / "data" / "checkpoints"

FIGURES_DIR = PROJECT_ROOT / "data" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

FUSION_CLASSES = {"early": EarlyFusionModel, "intermediate": IntermediateFusionModel, "late": LateFusionModel}
TASKS = ["classification", "survival"]
NUM_TIME_BINS = 15

train_labels = pd.read_csv(LABELS_DIR / "train_labels.csv")
folds = sorted(train_labels["fold"].unique())
MAX_TIME = float(train_labels["time_to_follow-up/BCR"].max())
f"{len(train_labels)} labeled patients across {len(folds)} folds"

## Step 1 · Run inference over held-out slides

Same slide-level dataset shape as training (one row per patient-slide), but now used purely for
a forward pass -- no optimizer, no shuffling, and we keep `patient_id` around so predictions can
be grouped back together afterwards.

In [ ]:
class SlideLevelDataset(Dataset):
    """One row per (patient, slide) -- identical shape to notebook 04\'s training dataset."""
    def __init__(self, patient_ids: list[str], labels_df: pd.DataFrame):
        self.labels_df = labels_df.set_index("patient_id")
        self.rows = [
            (patient_id, wsi_path)
            for patient_id in patient_ids
            for wsi_path in sorted(WSI_FEATURE_DIR.glob(f"{patient_id}_*_features.npy"))
        ]

    def __len__(self) -> int:
        return len(self.rows)

    def __getitem__(self, idx: int) -> dict:
        patient_id, wsi_path = self.rows[idx]
        label = self.labels_df.loc[patient_id]
        mri_path = next(MRI_FEATURE_DIR.glob(f"{patient_id}_*_features.npy"))
        clinical_path = CLINICAL_EMBEDDING_DIR / f"{patient_id}_embedding.npy"
        return {
            "patient_id": patient_id,
            "wsi": torch.from_numpy(np.load(wsi_path)).float(),
            "mri": torch.from_numpy(np.load(mri_path)).float(),
            "clinical": torch.from_numpy(np.load(clinical_path)).float(),
            "y_time": torch.tensor(label["time_to_follow-up/BCR"], dtype=torch.float32),
            "y_event": torch.tensor(label["BCR"], dtype=torch.float32),
        }


def collate_slides(samples: list[dict]) -> dict:
    return {
        "patient_id": [s["patient_id"] for s in samples],
        "wsi": [s["wsi"] for s in samples],
        "mri": torch.stack([s["mri"] for s in samples]),
        "clinical": torch.stack([s["clinical"] for s in samples]),
        "y_time": torch.stack([s["y_time"] for s in samples]),
        "y_event": torch.stack([s["y_event"] for s in samples]),
    }


@torch.no_grad()
def run_inference(model, loader, task: str) -> pd.DataFrame:
    """One row per slide: patient_id, ground truth, and the model\'s raw output for that slide."""
    model.eval()
    rows = []
    for batch in loader:
        wsi = [w.to(DEVICE) for w in batch["wsi"]]
        out = model({"wsi": wsi, "mri": batch["mri"].to(DEVICE), "clinical": batch["clinical"].to(DEVICE)})
        for i, patient_id in enumerate(batch["patient_id"]):
            row = {"patient_id": patient_id, "y_time": batch["y_time"][i].item(), "y_event": batch["y_event"][i].item()}
            if task == "classification":
                row["slide_prob"] = torch.sigmoid(out[i]).item()
            else:
                hazards = torch.sigmoid(out[i])
                survival = torch.cumprod(1 - hazards, dim=0)
                row["slide_risk"] = -survival.sum().item()  # higher survival sum -> lower risk
            rows.append(row)
    return pd.DataFrame(rows)

In [ ]:
# Try it on one (fusion, task, fold) checkpoint first.
fusion_name, task, fold = "early", "classification", folds[0]
ckpt_path = CHECKPOINT_DIR / f"{fusion_name}_{task}_fold{fold}.pt"

ckpt = torch.load(ckpt_path, map_location=DEVICE)
model = FUSION_CLASSES[fusion_name](task=task, num_time_bins=NUM_TIME_BINS).to(DEVICE)
model.load_state_dict(ckpt["model_state_dict"])

val_patients = train_labels.loc[train_labels["fold"] == fold, "patient_id"].tolist()
val_loader = DataLoader(SlideLevelDataset(val_patients, train_labels), batch_size=8, collate_fn=collate_slides)

slide_preds = run_inference(model, val_loader, task)
slide_preds.head()

## Step 2 · Aggregate slide-level predictions to patient level

A patient's slides were trained as independent rows, so their predictions need to be merged back
together for a patient-level metric to mean anything. The aggregation itself is deliberately
simple and happens on **probabilities / risk scores, not raw logits**:

- **Classification** -- mean predicted probability across a patient\'s slides, plus a majority
  vote for the hard label.
- **Survival** -- mean predicted risk score across a patient\'s slides.

In [ ]:
def aggregate_classification(slide_preds: pd.DataFrame) -> pd.DataFrame:
    grouped = slide_preds.groupby("patient_id").agg(
        y_event=("y_event", "first"),
        mean_prob=("slide_prob", "mean"),
        majority_vote=("slide_prob", lambda p: float((p > 0.5).mean() > 0.5)),
    )
    return grouped.reset_index()


def aggregate_survival(slide_preds: pd.DataFrame) -> pd.DataFrame:
    grouped = slide_preds.groupby("patient_id").agg(
        y_time=("y_time", "first"),
        y_event=("y_event", "first"),
        mean_risk=("slide_risk", "mean"),
    )
    return grouped.reset_index()


patient_preds = aggregate_classification(slide_preds)
patient_preds.head()

## Step 3 · Metrics — AUROC and C-index

In [ ]:
def evaluate_classification(patient_preds: pd.DataFrame) -> float:
    return roc_auc_score(patient_preds["y_event"], patient_preds["mean_prob"])


def evaluate_survival(patient_preds: pd.DataFrame) -> float:
    return concordance_index(
        patient_preds["y_time"].to_numpy(),
        patient_preds["mean_risk"].to_numpy(),
        patient_preds["y_event"].to_numpy(),
    )


f"Fold {fold} AUROC ({fusion_name}): {evaluate_classification(patient_preds):.3f}"

## Step 4 · Every fusion strategy, out-of-fold

Repeating the same load -> infer -> aggregate -> evaluate pipeline for every (fusion, task, fold)
combination, but scoring each fold\'s checkpoint **only on its own held-out validation patients**
(out-of-fold) before combining every fold\'s patients into one overall metric per (fusion, task).

In [ ]:
def load_checkpoint(fusion_name: str, task: str, fold: int):
    ckpt = torch.load(CHECKPOINT_DIR / f"{fusion_name}_{task}_fold{fold}.pt", map_location=DEVICE)
    model = FUSION_CLASSES[fusion_name](task=task, num_time_bins=NUM_TIME_BINS).to(DEVICE)
    model.load_state_dict(ckpt["model_state_dict"])
    return model


results = []
oof_patient_preds = {}  # (fusion, task) -> combined out-of-fold patient-level predictions

for fusion_name in FUSION_CLASSES:
    for task in TASKS:
        fold_dfs = []
        for fold in folds:
            model = load_checkpoint(fusion_name, task, fold)
            val_patients = train_labels.loc[train_labels["fold"] == fold, "patient_id"].tolist()
            val_loader = DataLoader(SlideLevelDataset(val_patients, train_labels), batch_size=8, collate_fn=collate_slides)
            slide_preds = run_inference(model, val_loader, task)
            agg = aggregate_classification(slide_preds) if task == "classification" else aggregate_survival(slide_preds)
            fold_dfs.append(agg)

        combined = pd.concat(fold_dfs, ignore_index=True)
        oof_patient_preds[(fusion_name, task)] = combined
        metric = evaluate_classification(combined) if task == "classification" else evaluate_survival(combined)
        metric_name = "AUROC" if task == "classification" else "C-index"
        results.append({"fusion": fusion_name, "task": task, "metric": metric_name, "value": metric})

comparison_df = pd.DataFrame(results)
comparison_df.pivot(index="fusion", columns="task", values="value")

## Post-processing — decision thresholds &amp; risk stratification

The raw model output isn\'t the final answer a clinician would look at:

- **Classification** -- a mean probability becomes a hard yes/no decision only after applying a
  threshold (0.5 by default, though this could be tuned per fusion strategy).
- **Survival** -- a continuous risk score becomes clinically actionable once patients are split
  into discrete risk groups. Here, the top 20% highest-risk patients become "high risk" and the
  rest "low risk" -- the same quantile-based split used to build the Kaplan-Meier curves below.

In [ ]:
def threshold_classification(patient_preds: pd.DataFrame, threshold: float = 0.5) -> pd.DataFrame:
    out = patient_preds.copy()
    out["predicted_label"] = (out["mean_prob"] >= threshold).astype(int)
    return out


def add_tiny_noise(x: np.ndarray, seed: int = 42) -> np.ndarray:
    """Breaks ties in the quantile split -- otherwise many patients can land exactly on the
    threshold when risk scores repeat."""
    rng = np.random.default_rng(seed)
    return x + 1e-8 * rng.standard_normal(size=x.shape)


def stratify_by_risk(patient_preds: pd.DataFrame, q: float = 0.2) -> pd.DataFrame:
    """Top-`q` fraction of patients by risk score become the high-risk group."""
    out = patient_preds.copy()
    risk = add_tiny_noise(out["mean_risk"].to_numpy())
    threshold_value = np.quantile(risk, 1 - q)
    out["risk_group"] = np.where(risk >= threshold_value, "high", "low")
    return out


example_survival = oof_patient_preds[("early", "survival")]
stratify_by_risk(example_survival).groupby("risk_group").size()

## Step 5 · Evaluation &amp; Visualization

### Comparing fusion strategies

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, task, metric_name in zip(axes, TASKS, ["AUROC", "C-index"]):
    subset = comparison_df[comparison_df["task"] == task]
    ax.bar(subset["fusion"], subset["value"], color=["#4FA3F0", "#7ec9a3", "#e0a03a"])
    ax.set_title(f"{task.capitalize()} ({metric_name})")
    ax.set_ylim(0.4, 1.0)
    ax.axhline(0.5, color="gray", linestyle="--", linewidth=1)
    ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "fusion_comparison.png", dpi=150)

### Kaplan-Meier risk stratification

For the best-performing survival model, split out-of-fold patients into high/low risk groups by
the same quantile rule from post-processing, then compare their survival curves with a log-rank
test.

In [ ]:
def km_with_ci(times, events, timeline):
    kmf = KaplanMeierFitter()
    kmf.fit(times, events)
    surv = kmf.survival_function_at_times(timeline).to_numpy()
    ci = kmf.confidence_interval_.reindex(kmf.confidence_interval_.index.union(timeline)).sort_index().ffill().reindex(timeline)
    return surv, ci.iloc[:, 0].to_numpy(), ci.iloc[:, 1].to_numpy()


best_fusion = comparison_df[comparison_df["task"] == "survival"].sort_values("value", ascending=False).iloc[0]["fusion"]
stratified = stratify_by_risk(oof_patient_preds[(best_fusion, "survival")])
low, high = stratified[stratified["risk_group"] == "low"], stratified[stratified["risk_group"] == "high"]

timeline = np.linspace(0, stratified["y_time"].max(), 15)
surv_lo, lo_lo, lo_hi = km_with_ci(low["y_time"], low["y_event"], timeline)
surv_hi, hi_lo, hi_hi = km_with_ci(high["y_time"], high["y_event"], timeline)
p_value = logrank_test(low["y_time"], high["y_time"], low["y_event"], high["y_event"]).p_value

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(timeline, surv_lo, color="#4FA3F0", linewidth=2, label="Low risk")
ax.fill_between(timeline, lo_lo, lo_hi, color="#4FA3F0", alpha=0.2)
ax.plot(timeline, surv_hi, color="#e05a5a", linewidth=2, label="High risk")
ax.fill_between(timeline, hi_lo, hi_hi, color="#e05a5a", alpha=0.2)
ax.set_xlabel("Time (months)"); ax.set_ylabel("Survival probability")
ax.set_title(f"{best_fusion.capitalize()} fusion — log-rank p = {p_value:.2e}")
ax.legend(); ax.grid(alpha=0.3)
fig.savefig(FIGURES_DIR / "km_stratification.png", dpi=150, bbox_inches="tight")

### ABMIL attention — which patches drove the prediction?

The same attention weights ABMIL used to pool a WSI\'s patches into one vector double as an
interpretability signal: overlay them back onto the slide thumbnail as a heatmap.

In [ ]:
@torch.no_grad()
def extract_abmil_attention(model, wsi_features: torch.Tensor) -> np.ndarray:
    attn_scores = model.abmil.attention(wsi_features.to(DEVICE))
    attn_weights = torch.softmax(attn_scores, dim=0)
    return attn_weights.squeeze(1).cpu().numpy()


def visualize_abmil_attention(patient_id: str, slide_id: str, attention: np.ndarray, coords: np.ndarray, patch_size: int = 224):
    wsi_path = RAW_WSI_DIR / patient_id / f"{patient_id}_{slide_id}.tif"
    slide = openslide.OpenSlide(str(wsi_path))
    width, height = slide.level_dimensions[0]
    scale = 1500 / max(width, height)
    thumb = slide.get_thumbnail((int(width * scale), int(height * scale)))

    heat = np.zeros((thumb.height, thumb.width), dtype=np.float32)
    for (x, y), score in zip(coords, attention):
        xt, yt = int(x * scale), int(y * scale)
        heat[yt:yt + int(patch_size * scale), xt:xt + int(patch_size * scale)] = score
    heat = gaussian_filter(heat, sigma=max(1, int(max(heat.shape) * 0.01)))
    heat = heat / heat.max() if heat.max() > 0 else heat

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.imshow(thumb); ax.imshow(heat, cmap="jet", alpha=0.5, vmin=0, vmax=1); ax.axis("off")
    ax.set_title(f"ABMIL attention — patient {patient_id}")
    fig.savefig(FIGURES_DIR / f"{patient_id}_abmil_attention.png", dpi=150, bbox_inches="tight")
    slide.close()


# example_patient_id, example_slide_id = "1003", "1"
# manifest = np.load(PREPARED_DIR / "wsi_patch_manifests" / f"{example_patient_id}_{example_slide_id}_patches.npz")
# features = torch.from_numpy(np.load(WSI_FEATURE_DIR / f"{example_patient_id}_{example_slide_id}_features.npy")).float()
# visualize_abmil_attention(example_patient_id, example_slide_id, extract_abmil_attention(model, features), manifest["coords"])

### Encoder-internal attention

Beyond ABMIL\'s patch-level weighting, each per-modality encoder has its own internal
transformer attention -- useful for asking *why* a single patch or MRI region looked important
in the first place. A forward hook on the last transformer block\'s attention module recovers
these weights without changing the model itself.

In [ ]:
@torch.no_grad()
def wsi_encoder_attention(wsi_model, patch_tensor: torch.Tensor, layer_idx: int = -1) -> np.ndarray:
    """CLS-token-to-patch attention from one of H-optimus-0\'s transformer blocks."""
    captured = {}

    def hook(module, inp, out):
        x = inp[0]
        B, N, C = x.shape
        qkv = module.qkv(x).reshape(B, N, 3, module.num_heads, C // module.num_heads).permute(2, 0, 3, 1, 4)
        q, k, _ = qkv.unbind(0)
        attn = (q @ k.transpose(-2, -1) * module.scale).softmax(dim=-1)
        captured["attn"] = attn.detach()

    handle = wsi_model.blocks[layer_idx].attn.register_forward_hook(hook)
    try:
        wsi_model(patch_tensor.unsqueeze(0).to(DEVICE))
    finally:
        handle.remove()

    cls_to_patches = captured["attn"][0, :, 0, 1:].mean(dim=0)  # average over heads
    grid = int(cls_to_patches.shape[0] ** 0.5)
    return cls_to_patches.reshape(grid, grid).cpu().numpy()


@torch.no_grad()
def mri_encoder_attention(mri_model, combined_input: torch.Tensor, layer_idx: int = -1) -> np.ndarray:
    """Patch-token attention from one of MRI-PTPCa\'s transformer layers (see src/mri_ptpca.py)."""
    captured = {}

    def hook(module, module_input, module_output):
        x_norm = module.norm(module_input[0])
        q, k, _ = module.to_qkv(x_norm).chunk(3, dim=-1)
        B, N, _ = q.shape
        head, dim_head = module.heads, q.shape[-1] // module.heads
        q = q.reshape(B, N, head, dim_head).permute(0, 2, 1, 3)
        k = k.reshape(B, N, head, dim_head).permute(0, 2, 1, 3)
        attn = module.attend((q @ k.transpose(-1, -2)) * module.scale)
        captured["attn"] = attn.detach()

    attn_module = mri_model.myViT_MM.transformer.layers[layer_idx][0]
    handle = attn_module.register_forward_hook(hook)
    try:
        mri_model(combined_input.to(DEVICE))
    finally:
        handle.remove()

    return captured["attn"][0].mean(dim=0).cpu().numpy()  # (N, N), averaged over heads

## Summary

This notebook closed the loop the course opened in notebook 04: loading trained checkpoints,
running inference over held-out slides, and aggregating slide-level output back to the patient
level the model was never directly optimized on. From there, post-processing turned continuous
outputs into decisions and risk groups, and the final visualizations -- fusion-strategy
comparison, Kaplan-Meier stratification, and two levels of attention (ABMIL and each encoder\'s
internal transformer attention) -- make those decisions inspectable rather than opaque.